In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
import numpy as np, pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, TimeSeriesSplit, GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from helper import load_clean_data, get_preprocessor, evaluate_models_cv, get_preprocessor_with_exclude, get_preprocessor_scaled
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import loguniform
from sklearn.metrics import roc_auc_score, f1_score, classification_report

SEED = 42


In [26]:
df = load_clean_data()
X, y = df.drop(columns=["y"]), df["y"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
tscv = TimeSeriesSplit(5)

In [27]:
best_pipe = Pipeline([
    ("prep", get_preprocessor_scaled()),
    ("model", LogisticRegression(random_state=SEED, max_iter=1000, solver="liblinear"))
])

initial_metrics = evaluate_models_cv([("logreg", LogisticRegression(random_state=SEED, max_iter=1000, solver="liblinear"))],
                                     X_train, y_train, get_preprocessor_scaled(), tscv)


/Users/karimildarovic/Задание_классификация/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karimildarovic/Задание_классификация/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/karimildarovic/Задание_классификация/venv/lib/python3.9/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf

1) GridSearch по параметрам логрега

In [28]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"],
}

grid = GridSearchCV(best_pipe, param_grid, scoring="roc_auc", cv=tscv, n_jobs=-1)
grid.fit(X_train, y_train)
grid.best_params_

{'model__C': 10, 'model__class_weight': None, 'model__penalty': 'l1'}

Оптимальные значения гиперпараметров: C=10, penalty='l1', class_weight=None

In [29]:
print("ROC-AUC до тюнинга:", initial_metrics.loc["logreg", "roc_auc"].round(3))
print("ROC-AUC после тюнинга:", round(grid.best_score_, 3))
print("прирост:", round(grid.best_score_ - initial_metrics.loc["logreg", "roc_auc"], 3))

ROC-AUC до тюнинга: 0.548
ROC-AUC после тюнинга: 0.562
прирост: 0.014


2) RandomizedSearch 

In [30]:
param_random = {
    "model__C": loguniform(1e-3, 1e2),
    "model__penalty": ["l1", "l2"],
    "model__class_weight": [None, "balanced"],
}

rand = RandomizedSearchCV(
    best_pipe, param_random,
    n_iter=50,
    scoring="roc_auc",
    cv=tscv,
    n_jobs=-1,
    random_state=SEED,
)

rand.fit(X_train, y_train)
rand.best_params_

{'model__C': np.float64(27.293781650374736),
 'model__class_weight': None,
 'model__penalty': 'l1'}

In [31]:
print("ROC-AUC до тюнинга:", initial_metrics.loc["logreg", "roc_auc"].round(3))
print("ROC-AUC после тюнинга:", round(rand.best_score_, 3))
print("прирост:", round(rand.best_score_ - initial_metrics.loc["logreg", "roc_auc"], 3))

ROC-AUC до тюнинга: 0.548
ROC-AUC после тюнинга: 0.562
прирост: 0.014


Протестированы два метода подбора: GridSearchCV и RandomizedSearchCV. Оба дали одинаковый ROC-AUC 0.562 (прирост +0.014 к baseline), сойдясь на penalty='l1', class_weight=None при C в районе 10–27. Совпадение результатов двух методов подтверждает, что потолок ограничен структурой данных. 

Финальная модель

In [32]:
final_model = grid.best_estimator_
y_pred  = final_model.predict(X_test)
y_probs = final_model.predict_proba(X_test)[:, 1]
print("ROC-AUC (test):", round(roc_auc_score(y_test, y_probs), 3))
print(classification_report(y_test, y_pred))

ROC-AUC (test): 0.716
              precision    recall  f1-score   support

           0       0.72      0.93      0.81      5370
           1       0.59      0.22      0.32      2509

    accuracy                           0.70      7879
   macro avg       0.65      0.57      0.56      7879
weighted avg       0.68      0.70      0.65      7879



ВЫВОД

На тесте ROC-AUC 0.716, то есть модель хорошо ранжирует объекты. При стандартном пороге 0.5 она смещена в сторону мажоритарного класса: recall класса «да» всего 0.22 (ловит 22% согласий), т.к. обучалась на сильно несбалансированных данных. Класс «нет» предсказывается хорошо (recall 
0.93). Качество по классу «да» можно поднять снижением порога классификации. ROC-AUC 0.716 показывает, что ранжирующая способность для этого есть